# Part 1: Ingest, Validate, Clean, and Enrich

This notebook handles raw data validation with Python's `csv` module, pandas cleaning, NumPy feature checks, clinic weather enrichment via Open-Meteo, and exporting `appointments_clean.csv` plus `load_data.sql`.

In [ ]:
import csv
import json
import numpy as np
import pandas as pd
import requests

## Task 1.1 — Read and validate with the `csv` module

Checking basic type parsing before touching pandas. Rows with blank or non-numeric values in `age`, `fee_inr`, `prior_no_shows`, or `distance_km` go into `rejected_rows.json`.

In [ ]:
def parse_row(row):
    parsed = dict(row)
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

accepted = []
rejected = []

with open("data/appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    total_rows = 0
    for row in reader:
        total_rows += 1
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

print(f"Total rows read: {total_rows}")
print(f"Accepted: {len(accepted)} | Rejected: {len(rejected)}")

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

## Task 1.2 — Inspect and clean (Pass 1)

Reloading directly from `data/appointments.csv` into pandas so we handle duplicates, inconsistent channel casing, date parsing, and missing targets in order.

In [ ]:
df = pd.read_csv("data/appointments.csv")
print("Raw shape:", df.shape)
print("\nInfo:")
df.info()
print("\nMissing values:\n", df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())

In [ ]:
# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print("After dropping duplicates:", len(df))

# (b) Standardise booking_channel
print("Raw booking_channel values:", df["booking_channel"].nunique())
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Cleaned booking_channel values:", df["booking_channel"].unique().tolist())

# (c) Parse dates
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After date parsing:", len(df))

# (d) Drop rows missing appt_date or status
df = df.dropna(subset=["appt_date", "status"]).reset_index(drop=True)
print("After dropping missing appt_date/status:", len(df))

## Task 1.3 — Validity, imputation, and outlier capping (Pass 2)

Fixing impossible ages first, imputing medians for `age` and `distance_km`, then capping IQR outliers on `distance_km` and `fee_inr`.

In [ ]:
# (a) Ages < 1 or > 100 are invalid
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid age count:", int(invalid_age_mask.sum()))
df.loc[invalid_age_mask, "age"] = np.nan

# (b) Impute medians
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print(f"Median age: {age_median}, Median distance_km: {dist_median}")

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) IQR outlier capping for distance_km and fee_inr
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    capped_count = int(((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, upper={upper}, capped={capped_count}")

# (d) Final rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Max fee_inr after capping:", df["fee_inr"].max())

## Task 1.4 — Feature engineering and NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = np.where(df["status"] == "No-Show", 1, 0)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"No-shows: {df['no_show'].sum()} / {len(df)} ({df['no_show'].mean() * 100:.1f}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Column means
col_means = np.mean(X, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# (b) Lead days > 14
lead_mask = X[:, 3] > 14
print(f"lead_days > 14: {lead_mask.sum()} rows ({lead_mask.mean() * 100:.1f}%)")

# (c) Z-scores for distance_km (population std, NumPy default)
dist_arr = X[:, 1]
z_dist = (dist_arr - np.mean(dist_arr)) / np.std(dist_arr)
print("Rows with |z| > 2 for distance_km:", int((np.abs(z_dist) > 2).sum()))

## Task 1.5 — Enrich clinics using Open-Meteo

Fetching the midnight-UTC `temperature_2m` reading for each clinic's coordinates with one retry on failure.

In [ ]:
clinics_df = pd.read_csv("data/clinics.csv")
temps = []

for _, row in clinics_df.iterrows():
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    temp_val = None
    last_err = ""

    for attempt in range(2):
        try:
            resp = requests.get(url, params=params, timeout=10)
            if resp.status_code == 200:
                payload = resp.json()
                hourly = payload.get("hourly", {})
                t_list = hourly.get("temperature_2m", [])
                if t_list:
                    temp_val = t_list[0]
                    break
                last_err = "missing hourly.temperature_2m"
            else:
                last_err = f"HTTP {resp.status_code}"
        except requests.exceptions.RequestException as e:
            last_err = str(e)

    if temp_val is None:
        print(f"Failed for {row['clinic_name']}: {last_err}")
    temps.append(temp_val)

clinics_df["temp_c"] = temps
clinics_df.to_csv("clinics_enriched.csv", index=False)
clinics_df

## Task 1.6 — Export clean table and SQL load script

In [ ]:
cols_order = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
df = df[cols_order]

print("appointment_id unique:", df["appointment_id"].is_unique, "| count:", df["appointment_id"].nunique())
print("Total missing values across 16 columns:", int(df.isnull().sum().sum()))

df.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv with shape:", df.shape)

with open("load_data.sql", "w", encoding="utf-8") as f:
    # Clinics first (parent table for foreign key)
    for _, r in clinics_df.iterrows():
        temp_sql = "NULL" if pd.isna(r["temp_c"]) else f"{float(r['temp_c'])}"
        stmt = (
            f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}',"
            f"'{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{temp_sql});\n"
        )
        f.write(stmt)

    for _, r in df.iterrows():
        stmt = (
            f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}',"
            f"'{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{int(r['age'])},"
            f"'{r['gender']}',{float(r['distance_km'])},{int(r['fee_inr'])},"
            f"'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',"
            f"{int(r['prior_no_shows'])},'{r['status']}',{int(r['lead_days'])},{int(r['no_show'])});\n"
        )
        f.write(stmt)

print("Wrote load_data.sql successfully.")

### Quick check for Part 2.2 (Python groupby verification)

Confirming clinic-level appointment and no-show counts match the SQL query.

In [ ]:
clinic_check = (
    df.merge(clinics_df[["clinic_id", "clinic_name", "city"]], on="clinic_id")
      .groupby(["clinic_name", "city"], as_index=False)
      .agg(appointments=("appointment_id", "count"), no_shows=("no_show", "sum"))
)
clinic_check["no_show_pct"] = (clinic_check["no_shows"] / clinic_check["appointments"] * 100).round(1)
clinic_check.sort_values("no_show_pct", ascending=False)